<a href="https://colab.research.google.com/github/Daniel-534/MecanicaCeleste/blob/main/Modelacion3/CRTBP_Jupiter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#





*   Soleil Dayana Niño Murcia
*   Daniel Soto Villada
*   Sara Calle Muñoz





https://colab.research.google.com/github/Daniel-534/MecanicaCeleste/blob/main/Modelacion3/CRTBP_Jupiter.ipynb

In [82]:
!pip install -Uq pymcel celluloid

In [83]:
import pymcel as pc
import numpy as np
import matplotlib.pyplot as plt
from celluloid import Camera
from IPython.display import HTML, display

### Consulta de Efemérides con Horizons

Para inicializar la simulación del sistema **Júpiter–asteroides troyanos–misión Lucy**, es necesario obtener las condiciones iniciales precisas (posición y velocidad) de cada cuerpo. Utilizamos la función `pc.consulta_horizons` para extraer los vectores de estado desde la base de datos JPL Horizons.

1. **Definimos el intervalo temporal** (`Epochs`) de **un año** (del 16 de junio de 2026 al 16 de junio de 2027) con pasos de **5 días** (`step='5d'`).
2. **Especificamos los identificadores de los cuerpos** a consultar:
   - **Sol** (`10`): cuerpo central del sistema.
   - **Júpiter** (`599`): planeta dominante gravitacionalmente.
   - **624 Hektor** (`DES=2000624;`): asteroide troyano de Júpiter (campo troyano, L5).
   - **153 Hilda** (`DES=2000153`): asteroide del grupo de Hilda (resonancia 3:2 con Júpiter).
   - **617 Patroclus** (`DES=2000617;`): asteroide troyano binario (campo troyano, L5).
   - **Lucy** (`-49`): sonda espacial de la NASA en ruta hacia los troyanos.
3. **Iteramos sobre cada cuerpo** para descargar sus tablas de efemérides, tiempos en día juliano (`tiempo_jd`) y vectores de estado (`vector_estado`), almacenándolos en diccionarios (`all_tablas`, `all_tiempo_jd`, `all_vector_estado`) para su posterior procesamiento y transformación al sistema de referencia adecuado.

In [84]:
Epochs = {'start': '2026-06-16', 'stop': '2027-06-16', 'step': '5d'}

cuerpos = {
    "Sol": "10",                        # 1
    "Jupiter": "599",                   # 2
    "624_Hector": "DES=2000624;",       # 3
    "153_Hilda": "DES=2000153",         # 4
    "617_Patroclus": "DES=2000617",     # 5
    "Lucy": "-49"                       # 6
}

all_tablas = {}
all_tiempo_jd = {}
all_vector_estado = {}

for name, body_id in cuerpos.items():
  tabla_body, tiempo_jd_body, vector_estado_body = pc.consulta_horizons(
      id=body_id,
      location='@0',
      epochs=Epochs,
      datos='vectors',
  )

  all_tablas[name] = tabla_body
  all_tiempo_jd[name] = tiempo_jd_body
  all_vector_estado[name] = vector_estado_body

tabla = all_tablas
tiempo_jd = all_tiempo_jd
vector_estado = all_vector_estado

In [85]:
pos_sol = np.linalg.norm(vector_estado["Sol"][["x","y","z"]].iloc[0].to_numpy())
pos_jupiter = np.linalg.norm(vector_estado["Jupiter"][["x","y","z"]].iloc[0].to_numpy())
dist = pos_jupiter - pos_sol
print(dist)

787157329573.4178


# Configuración del CRTBP para el sistema Sol-Júpiter

## Descripción del sistema

Para estudiar la dinámica de los asteroides troyanos de Júpiter (**624 Hektor**, **153 Hilda**, **617 Patroclus**) y de la sonda espacial **Lucy** bajo la influencia gravitacional conjunta del Sol ($m_1$) y Júpiter ($m_2$), asumimos que las órbitas de los cuerpos primarios son **circulares** alrededor de su centro de masas común. Este es el escenario clásico del **Problema Circular Restringido de los Tres Cuerpos (CRTBP)**, descrito en la **Sección 8.3** del libro de Zuluaga.

En esta aproximación, las masas de los cuerpos terciarios (asteroides y sonda) son despreciables comparadas con las de los primarios, de modo que no perturban el movimiento de Sol y Júpiter:

| Cuerpo | Identificador | Masa (kg) | Rol |
|--------|---------------|-----------|-----|
| Sol | $m_1$ | $M_\odot$ | Primario (cuerpo central) |
| Júpiter | $m_2$ | $M_J$ | Secundario |
| 624 Hektor | $m_3$ | $7.91 \times 10^{18}$ | Partícula de prueba (troyano L5) |
| 153 Hilda | $m_4$ | $5.20 \times 10^{18}$ | Partícula de prueba (resonancia 3:2) |
| 617 Patroclus | $m_5$ | $1.36 \times 10^{18}$ | Partícula de prueba (troyano L5) |
| Lucy | $m_6$ | $1.55 \times 10^{3}$ | Partícula de prueba (sonda) |

## Unidades canónicas

Para simplificar las ecuaciones de movimiento, utilizamos un sistema de **unidades canónicas** (Sección 8.4, Zuluaga) donde:

- **Unidad de masa:** $U_M = m_1 + m_2$ (masa total del sistema Sol-Júpiter).
- **Unidad de longitud:** $U_L = a = 787\,157\,329\,573.4178 \text{ m}$ (semieje mayor de la órbita de Júpiter respecto al Sol, es decir, la distancia constante entre ambos primarios).
- **Unidad de tiempo:** $U_T = \sqrt{\dfrac{U_L^3}{G \, U_M}}$, definida tal que la velocidad angular media del sistema sea $n = 1$.
- **Unidad de velocidad:** $U_V = U_L / U_T$.

En este sistema, la constante de gravitación universal toma el valor $G = 1$, y la velocidad angular del sistema rotante coincide con la velocidad angular media de los primarios.

## Parámetros de masa adimensionales

Siguiendo la notación de la **Sección 8.4** de Zuluaga, definimos el parámetro de masa:

$$
\alpha \equiv \mu_2 = \frac{G \, m_2}{G(m_1 + m_2)} = \frac{m_2}{m_1 + m_2}
$$

que representa la fracción de masa del cuerpo secundario (Júpiter). La masa reducida del cuerpo primario (Sol) es entonces:

$$
\mu_1 = 1 - \alpha = \frac{m_1}{m_1 + m_2}
$$

Con los valores reales del Sistema Solar ($M_\odot \approx 1.989 \times 10^{30}$ kg y $M_J \approx 1.898 \times 10^{27}$ kg), se obtiene:

$$
\alpha \approx 9.547 \times 10^{-4}
$$

## Posiciones de los primarios en el sistema rotante

En el sistema de referencia rotante (que gira con velocidad angular $\omega = n = 1$ en unidades canónicas), los cuerpos primarios permanecen fijos sobre el eje $x$, tal como se describe en la **Sección 8.4** de Zuluaga:

- **Sol (primario):** ubicado en $x_1 = -\alpha$, $y_1 = 0$, $z_1 = 0$.
- **Júpiter (secundario):** ubicado en $x_2 = 1 - \alpha$, $y_2 = 0$, $z_2 = 0$.

## Ecuaciones de movimiento

La dinámica de cada partícula de prueba (asteroide o sonda) en el sistema rotante está gobernada por la ecuación (8.7) de Zuluaga:

$$
\ddot{\vec{r}} = -\frac{(1-\alpha)}{r_1^3}\vec{r}_1 - \frac{\alpha}{r_2^3}\vec{r}_2 - \hat{e}_z \times (\hat{e}_z \times \vec{r}) - 2\hat{e}_z \times \dot{\vec{r}}
$$

donde:

- $\vec{r}_1 = (x + \alpha)\hat{e}_x + y\hat{e}_y + z\hat{e}_z$ es la posición relativa al Sol.
- $\vec{r}_2 = (x - 1 + \alpha)\hat{e}_x + y\hat{e}_y + z\hat{e}_z$ es la posición relativa a Júpiter.
- El tercer término corresponde a la **aceleración centrífuga**.
- El cuarto término corresponde a la **aceleración de Coriolis**.

## Constante de Jacobi

El CRTBP posee una única integral primera de movimiento: la **constante de Jacobi** (Sección 8.7, Zuluaga),

$$
C_J = \frac{2(1-\alpha)}{r_1} + \frac{2\alpha}{r_2} + (x^2 + y^2) - v^2
$$

que permite determinar las **regiones permitidas** del espacio (donde $v^2 \geq 0$) y las **superficies de cero velocidad** que actúan como barreras dinámicas para las partículas de prueba (Sección 8.11, Zuluaga). Esta constante es particularmente útil para analizar la estabilidad de los puntos de Lagrange $L_4$ y $L_5$, donde residen los asteroides troyanos como 624 Hektor y 617 Patroclus.

## Observación importante

Dado que $\alpha \ll 1$, los puntos de Lagrange triangulares $L_4$ y $L_5$ (ubicados a $60°$ adelante y detrás de Júpiter en su órbita) son estables según el criterio de Routh, lo que explica la existencia de los **asteroides troyanos** y la capacidad de la misión Lucy para visitarlos. El asteroide **153 Hilda**, al estar en resonancia 3:2 con Júpiter, requiere un análisis más allá del CRTBP estándar (teoría de perturbaciones), ya que su dinámica no corresponde a un equilibrio en los puntos de Lagrange.

In [86]:
G = pc.constantes.G
m1 = pc.constantes.M_sun   # Masa del Sol
m2 = pc.constantes.M_jup   # Masa de Jupiter

m3 = 7.91e18   # 624_Hector
m4 = 5.20e18   # 153_Hilda
m5 = 1.36e18   # 617_Patroclus
m6 = 1.55e3    # Lucy (Masa de lanzamiento)

a = 787157329573.4178

# Definimos unidades canonicas
U_M = m1+m2
U_L = a
U_T = np.sqrt(U_L**3/(G*U_M))
U_V = U_L/U_T

mu2 = m2*G / (G*U_M)
alpha = mu2
mu1 = 1-alpha

# Transformación al Marco de Referencia Rotante y Unidades Canónicas del CRTBP

## Descripción del procedimiento

Para estudiar la dinámica de una partícula de prueba bajo la influencia gravitacional conjunta de dos cuerpos primarios en el marco del **Problema Circular Restringido de Tres Cuerpos (CRTBP)**, es necesario transformar el vector de estado inicial desde un sistema de referencia inercial (en unidades del SI) al **sistema de referencia rotante** en unidades canónicas. En este marco, los cuerpos primarios permanecen fijos sobre el eje $x$ y la velocidad angular del sistema es $\omega = 1$.

Este procedimiento, fundamentado en la **Sección 8.3** (Configuración del CRTBP), **Sección 8.4** (Unidades canónicas) y **Sección 5.5** (Dinámica en sistemas no inerciales) del libro de Zuluaga, involucra una secuencia de transformaciones geométricas, traslaciones y adimensionalización.

## Procedimiento Analítico

### 1. Cálculo del Baricentro y Traslación

El primer paso consiste en determinar la posición y velocidad del centro de masa (baricentro) del sistema binario formado por los dos cuerpos primarios ($m_1$ y $m_2$). Según la **Sección 5.4.1** de Zuluaga, el baricentro se calcula como:

$$
\vec{R}_{CM} = \frac{m_1 \vec{r}_1 + m_2 \vec{r}_2}{m_1 + m_2}, \qquad \vec{V}_{CM} = \frac{m_1 \vec{v}_1 + m_2 \vec{v}_2}{m_1 + m_2}
$$

Posteriormente, se traslada el estado de la partícula de prueba (tercer cuerpo) al sistema de referencia centrado en el baricentro:

$$
\vec{r}_{rel} = \vec{r}_3 - \vec{R}_{CM}, \qquad \vec{v}_{rel} = \vec{v}_3 - \vec{V}_{CM}
$$

### 2. Rotación del Sistema de Coordenadas

En el CRTBP, el sistema de referencia rotante debe estar alineado de tal manera que los dos cuerpos primarios permanezcan fijos sobre el eje $x$. Para lograr esto, se calcula el ángulo $\theta$ que forma el vector relativo entre los cuerpos primarios con el eje $x$ del sistema inercial:

$$
\theta = \arctan2\left(r_{12,y},\, r_{12,x}\right)
$$

donde $\vec{r}_{12} = \vec{r}_2 - \vec{r}_1$. Se construye entonces la matriz de rotación $R(-\theta)$ que rota el sistema en sentido contrario al ángulo $\theta$, alineando el nuevo eje $x$ con la línea que une los primarios.

### 3. Adimensionalización (Unidades Canónicas)

Siguiendo la **Sección 8.4** de Zuluaga, se adopta un sistema de unidades canónicas donde:
*   **Unidad de longitud:** $U_L = a$ (distancia constante entre los primarios).
*   **Unidad de masa:** $U_M = m_1 + m_2$.
*   **Unidad de tiempo:** $U_T = \sqrt{U_L^3 / (G U_M)}$, de modo que la velocidad angular media del sistema sea $n = 1$.
*   **Unidad de velocidad:** $U_V = U_L / U_T$.

Las cantidades relativas se adimensionalizan dividiendo por sus respectivas unidades:

$$
\vec{r}_{can} = \frac{\vec{r}_{rel}}{U_L}, \qquad \vec{v}_{can} = \frac{\vec{v}_{rel}}{U_V}
$$

### 4. Transformación de la Velocidad en el Marco Rotante

La posición canónica rotada se obtiene aplicando la matriz de rotación: $\vec{r}_{rot} = R(-\theta) \vec{r}_{can}$.

Para la velocidad, es crucial aplicar la regla de transformación de velocidades en sistemas rotantes (deducida en la **Sección 5.5.3** de Zuluaga, Ec. 5.64). La velocidad en el marco rotante difiere de la velocidad inercial debido al movimiento de rotación de los ejes:

$$
\vec{v}_{rot} = \vec{v}_{inercial, rotado} - \vec{\omega} \times \vec{r}_{rot}
$$

En unidades canónicas, el vector de velocidad angular del sistema es $\vec{\omega} = (0, 0, 1)^T$. Por lo tanto, la velocidad final en el marco rotante canónico es:

$$
\vec{v}_{rot} = R(-\theta) \vec{v}_{can} - \hat{e}_z \times \vec{r}_{rot}
$$

El vector de estado final en el marco del CRTBP se ensambla como $\vec{Y} = (\vec{r}_{rot},\, \vec{v}_{rot})$.

---

## Observaciones Físicas y Teóricas

1.  **Velocidad angular unitaria:** En el sistema de unidades canónicas del CRTBP, la velocidad angular del marco rotante es $\omega = n = 1$ por construcción (Zuluaga, Sección 8.4). Esto simplifica enormemente las ecuaciones de movimiento, eliminando el parámetro $\omega$ explícito y haciendo que la aceleración centrífuga y de Coriolis dependan únicamente de las coordenadas canónicas.
2.  **Corrección de Coriolis en la transformación:** La resta del término $\vec{\omega} \times \vec{r}_{rot}$ en la transformación de la velocidad es físicamente obligatoria. Si se ignorara, la energía cinética calculada en el marco rotante sería incorrecta, ya que la velocidad inercial y la velocidad relativa al marco rotante no son la misma magnitud. Esta corrección garantiza que la **Constante de Jacobi** se conserve correctamente a partir del vector de estado inicial.
3.  **Configuración de equilibrio:** Al final de esta transformación, los cuerpos primarios $m_1$ y $m_2$ tendrán coordenadas fijas $(-\alpha, 0, 0)$ y $(1-\alpha, 0, 0)$ respectivamente, donde $\alpha = m_2 / (m_1 + m_2)$ es el parámetro de masa canónico. Esto cumple la condición fundamental del CRTBP descrita en la **Ec. (8.5)** de Zuluaga.

In [87]:
def transformar_a_CRTBP_frame(estado_si, m1_si, m2_si, U_L, U_V):

    # Validación y extracción dinámica de las 3 keys
    try:
        keys = list(estado_si.keys())
        if len(keys) != 3:
            raise ValueError(f"Se esperaban exactamente 3 keys en estado_si, pero se encontraron {len(keys)}.")

        # Desempaquetamos las 3 keys
        k1, k2, k3 = keys

    except ValueError as e:
        # Si falla la validación de longitud, relanzamos el error
        raise e
    except Exception as e:
        # Si estado_si no es un diccionario o falla al leer las keys
        raise TypeError(f"Error al procesar estado_si. Asegúrate de que sea un diccionario. Detalle: {e}")

    def get_vec(name):
        data = estado_si[name] # Usualmente esto es un dataframe
        if hasattr(data, 'values'): # Si es DataFrame
            return data[['x', 'y', 'z', 'vx', 'vy', 'vz']].iloc[0].to_numpy()
        else: # Si es array/lista
            return np.array(data[0]) # Asumiendo que es el primer instante

    r1_si = get_vec(k1)[:3]
    v1_si = get_vec(k1)[3:]

    r2_si = get_vec(k2)[:3]
    v2_si = get_vec(k2)[3:]

    r3_si = get_vec(k3)[:3]
    v3_si = get_vec(k3)[3:]

    # Calcular el Baricentro del sistema binario (m1 + m2)
    M_total = m1_si + m2_si

    r_cm_si = (m1_si * r1_si + m2_si * r2_si) / M_total
    v_cm_si = (m1_si * v1_si + m2_si * v2_si) / M_total

    # Trasladar al sistema centrado en el baricentro
    r_rel_si = r3_si - r_cm_si
    v_rel_si = v3_si - v_cm_si

    # Determinar el ángulo de rotación theta
    r12_si = r2_si - r1_si
    theta = np.arctan2(r12_si[1], r12_si[0])

    # Construir la Matriz de Rotación R(-theta)
    cos_t = np.cos(-theta)
    sin_t = np.sin(-theta)

    R = np.array([
        [cos_t, -sin_t, 0],
        [sin_t,  cos_t, 0],
        [0,      0,     1]
    ])

    # Aplicar la rotación a la posición relativa
    r_rot_si = np.dot(R, r_rel_si)

    r_can_inertial = r_rel_si / U_L
    v_can_inertial = v_rel_si / U_V

    # Ahora aplicamos la rotación a las cantidades canónicas
    r_can_rot = np.dot(R, r_can_inertial)

    # Corrección de velocidad por rotación del marco
    omega_can = np.array([0, 0, 1])
    omega_cross_r = np.cross(omega_can, r_can_rot)

    # Rotamos la velocidad inercial canónica
    v_can_inertial_rot = np.dot(R, v_can_inertial)

    # Velocidad final en el marco rotante canónico
    v_can_rot = v_can_inertial_rot - omega_cross_r

    # Ensamblar el vector de estado final
    Y_canonico = np.concatenate((r_can_rot, v_can_rot))

    return Y_canonico

### Preparación de las Condiciones Iniciales en el Marco Canónico del CRTBP

Para estudiar la dinámica de las partículas de prueba (los asteroides troyanos 624 Hektor, 153 Hilda, 617 Patroclus y la sonda Lucy) bajo la influencia gravitacional conjunta del Sol y Júpiter, es necesario expresar sus vectores de estado iniciales en el sistema de referencia rotante y en unidades canónicas propias del Problema Circular Restringido de Tres Cuerpos (CRTBP).

Dado que, por definición del CRTBP, las partículas de prueba tienen masas despreciables y no perturban el movimiento de los cuerpos primarios, el marco de referencia está determinado exclusivamente por el sistema binario Sol-Júpiter. En este marco, los cuerpos primarios permanecen fijos sobre el eje $x$ en las posiciones $x_1 = -\alpha$ y $x_2 = 1-\alpha$ (donde $\alpha = m_2 / (m_1 + m_2)$ es el parámetro de masa canónico), y la velocidad angular del sistema es $\omega = 1$.

El procedimiento implementado en el código consiste en iterar sobre cada partícula de prueba y, para cada una, extraer su vector de estado inercial junto con el de los cuerpos primarios. Estos tres estados se agrupan en un diccionario reducido (`estado_si_reducido`) que se pasa como argumento a la función de transformación `transformar_a_CRTBP_frame`. Esta función aplica de manera sistemática las transformaciones geométricas y cinemáticas deducidas en la **Sección 5.5** (Dinámica en sistemas de referencia no inerciales) y **Sección 8.4** (Las unidades canónicas del CRTBP) de Zuluaga:

1.  **Traslación al baricentro:** Se resta la posición y velocidad del centro de masa del sistema Sol-Júpiter.
2.  **Rotación del marco:** Se aplica una matriz de rotación $R(-\theta)$ para alinear el nuevo eje $x$ con la línea que une instantáneamente al Sol y a Júpiter.
3.  **Adimensionalización:** Las coordenadas y velocidades se dividen por las unidades canónicas de longitud ($U_L = a$) y velocidad ($U_V = U_L / U_T$).
4.  **Corrección por rotación del marco:** A la velocidad adimensional rotada se le resta el término $\vec{\omega} \times \vec{r}_{can}$ (con $\vec{\omega} = \hat{e}_z$ en unidades canónicas) para obtener la velocidad relativa al marco rotante. Esta corrección es físicamente obligatoria para garantizar que la energía cinética y, en consecuencia, la **Constante de Jacobi** (Sección 8.7, Zuluaga) se evalúen correctamente.

El resultado de este proceso es un conjunto de vectores de estado canónicos $\vec{Y} = (x, y, z, v_x, v_y, v_z)$ para cada partícula de prueba, almacenados en las variables `Ys_Hector`, `Ys_Hilda`, `Ys_Patroclus` y `Ys_Lucy`. Estos vectores constituyen las condiciones iniciales exactas requeridas para integrar numéricamente las ecuaciones de movimiento del CRTBP (Ec. 8.7 en Zuluaga) y analizar la estabilidad orbital, las regiones de exclusión y la evolución temporal de estos cuerpos en el potencial modificado del sistema Sol-Júpiter.

In [88]:
cuerpos_fijos = ['Sol', 'Jupiter']
cuerpos_a_calcular = ['624_Hector', '153_Hilda', '617_Patroclus', 'Lucy']

resultados_Ys = {}

for cuerpo_menor in cuerpos_a_calcular:

    keys_necesarias = cuerpos_fijos + [cuerpo_menor]
    estado_si_reducido = {k: vector_estado[k] for k in keys_necesarias}

    x, y, z, vx, vy, vz = transformar_a_CRTBP_frame(
        estado_si_reducido, m1, m2, U_L, U_V
    )

    nombre_resultado = f"Ys_{cuerpo_menor}"
    resultados_Ys[nombre_resultado] = [x, y, z, vx, vy, vz]

Ys_Hector   = resultados_Ys['Ys_624_Hector']
Ys_Hilda    = resultados_Ys['Ys_153_Hilda']
Ys_Patroclus = resultados_Ys['Ys_617_Patroclus']
Ys_Lucy     = resultados_Ys['Ys_Lucy']

### Integración numérica de las ecuaciones del CRTBP

Para estudiar la evolución dinámica de las partículas de prueba (624 Hektor, 153 Hilda, 617 Patroclus y la sonda Lucy), integramos numéricamente las ecuaciones de movimiento del Problema Circular Restringido de Tres Cuerpos (CRTBP) descritas en la **Sección 8.5** de Zuluaga.

Se define un intervalo de integración de $t = 0$ a $t = 300$ unidades de tiempo canónicas (equivalente a aproximadamente 300 períodos orbitales de Júpiter), con una resolución de 10,000 pasos para garantizar la precisión de las trayectorias. Utilizando la rutina `pc.crtbp_solucion`, propagamos el estado inicial de cada cuerpo (extraído de `resultados_Ys`) y obtenemos sus posiciones y velocidades tanto en el sistema de referencia rotante (donde Sol y Júpiter permanecen fijos) como en el sistema inercial.

In [89]:
ts = np.linspace(0, 300, 10000)

rs_Hector, vs_Hector, ris_Hector, vis_Hector, r1s_Hector, r2s_Hector = pc.crtbp_solucion(alpha, resultados_Ys['Ys_624_Hector'][:3], resultados_Ys['Ys_624_Hector'][3:], ts)

rs_Hilda, vs_Hilda, ris_Hilda, vis_Hilda, r1s_Hilda, r2s_Hilda = pc.crtbp_solucion(alpha, resultados_Ys['Ys_153_Hilda'][:3], resultados_Ys['Ys_153_Hilda'][3:], ts)

rs_Patroclus, vs_Patroclus, ris_Patroclus, vis_Patroclus, r1s_Patroclus, r2s_Patroclus = pc.crtbp_solucion(alpha, resultados_Ys['Ys_617_Patroclus'][:3], resultados_Ys['Ys_617_Patroclus'][3:], ts)

rs_Lucy, vs_Lucy, ris_Lucy, vis_Lucy, r1s_Lucy, r2s_Lucy = pc.crtbp_solucion(alpha, resultados_Ys['Ys_Lucy'][:3], resultados_Ys['Ys_Lucy'][3:], ts)

### Visualización de las trayectorias en el sistema rotante

Para analizar la dinámica de las partículas de prueba (624 Hektor, 153 Hilda, 617 Patroclus y la sonda Lucy) en el marco del Problema Circular Restringido de Tres Cuerpos (CRTBP), representamos gráficamente sus trayectorias en el plano del sistema de referencia rotante. En este sistema, los cuerpos primarios permanecen fijos sobre el eje $x$: el Sol en la posición $x_1 = -\alpha$ y Júpiter en $x_2 = 1-\alpha$, donde $\alpha$ es el parámetro de masa canónico. Las coordenadas espaciales se expresan en unidades canónicas ($U_L = a$, la distancia Sol-Júpiter), lo que permite visualizar directamente la estabilidad orbital, las libraciones alrededor de los puntos de equilibrio y la evolución temporal de estos cuerpos bajo el potencial modificado del sistema Sol-Júpiter.

In [ ]:
plt.figure(figsize=(6*1.5, 4*1.5))

plt.title('Trayectorias de asteroides troyanos y la sonda Lucy en el sistema Sol-Júpiter (CRTBP)', fontsize=14)
plt.plot(rs_Hector[:,0], rs_Hector[:,1], "-m", lw=1, label="Hector")
plt.plot(rs_Hilda[:,0], rs_Hilda[:,1], "-c", lw=1, label="Hilda")
plt.plot(rs_Patroclus[:,0], rs_Patroclus[:,1], "-k", lw=1, label="Patroclus")
plt.plot(rs_Lucy[:,0], rs_Lucy[:,1], "-g", lw=1, label="Lucy")

plt.plot(-alpha, 0, 'o', color='red', markersize=12, label='Sol')
plt.plot(1 - alpha, 0, 'o', color='blue', markersize=8, label='Jupiter')

plt.xlabel('x (unidades canónicas)', fontsize=12)
plt.ylabel('y (unidades canónicas)', fontsize=12)

plt.grid(ls="--")
plt.axis('equal')

plt.legend()
plt.tight_layout()
plt.savefig('fig_trayectorias_rotantes.png', dpi=300);

### Visualización de las trayectorias en el sistema inercial

A diferencia del marco rotante, en el sistema de referencia inercial los cuerpos primarios (Sol y Júpiter) describen órbitas circulares alrededor del baricentro común del sistema, el cual permanece en reposo en el origen. Las trayectorias de las partículas de prueba (624 Hektor, 153 Hilda, 617 Patroclus y la sonda Lucy) se grafican a partir de las posiciones `ris` obtenidas de la integración numérica, expresadas en unidades canónicas ($U_L = a$). En este marco, la complejidad aparente del movimiento de los asteroides y la sonda refleja la superposición de su órbita heliocéntrica con el movimiento orbital de Júpiter, lo que contrasta con la simplicidad de las libraciones observadas en el sistema rotante.

In [ ]:
plt.figure(figsize=(6*1.5, 4*1.5))
plt.title('Trayectorias de asteroides troyanos y la sonda Lucy en el sistema inercial (CRTBP)', fontsize=14)
plt.plot(ris_Hector[:,0], ris_Hector[:,1], "-m", lw=0.5, label="Hector")
plt.plot(ris_Hilda[:,0], ris_Hilda[:,1], "-c", lw=0.5, label="Hilda")
plt.plot(ris_Patroclus[:,0], ris_Patroclus[:,1], "-k", lw=0.5, label="Patroclus")
plt.plot(ris_Lucy[:,0], ris_Lucy[:,1], "-g", lw=0.5, label="Lucy")

plt.plot(r1s_Lucy[:,0], r1s_Lucy[:,1], color='red', linewidth=3, label='Sol')
plt.plot(r2s_Lucy[:,0], r2s_Lucy[:,1], color='blue', linewidth=3, label='Jupiter')


plt.xlabel('x (unidades canónicas)', fontsize=12)
plt.ylabel('y (unidades canónicas)', fontsize=12)

plt.grid(ls="--")
plt.axis('equal')

plt.legend()
plt.tight_layout()
plt.savefig('fig_trayectorias_inercial.png', dpi=300);

### Visualización tridimensional del potencial modificado del CRTBP

Para comprender la dinámica de una partícula de prueba en el Problema Circular Restringido de Tres Cuerpos (CRTBP), es fundamental analizar la estructura del **potencial modificado** ($V_{mod}$), el cual determina la aceleración que experimenta la partícula cuando se encuentra en reposo en el sistema de referencia rotante. Según la **Sección 8.12** del libro de Zuluaga, este potencial escalar se define como la suma del potencial gravitacional newtoniano de los dos cuerpos primarios y el potencial centrífugo asociado a la rotación del marco de referencia:

$$ V_{mod} = -\frac{1-\alpha}{r_1} - \frac{\alpha}{r_2} - \frac{1}{2}(x^2+y^2) $$

donde $\alpha$ es el parámetro de masa del sistema, y $r_1$ y $r_2$ son las distancias desde la partícula de prueba hasta el cuerpo primario más masivo (ubicado en $x_1 = -\alpha$) y el cuerpo secundario (ubicado en $x_2 = 1-\alpha$), respectivamente.

El código proporcionado implementa el cálculo de este potencial sobre una malla bidimensional en el plano orbital ($z=0$) y utiliza la biblioteca `Plotly` para generar una representación de superficie tridimensional.

Un detalle algorítmico crucial en la implementación es la introducción de un **factor de suavizado** (`soft = 0.03`) dentro de la expresión para las distancias relativas:
$$ r_i = \sqrt{(x-x_i)^2 + y^2 + z^2 + \text{soft}} $$
Como se discute en la **Nota: el factor de suavizado** de la Sección 8.12, el potencial teórico tiende a menos infinito cuando la partícula de prueba se ubica exactamente en la posición de uno de los cuerpos masivos ($r_1=0$ o $r_2=0$). Esta singularidad matemática no solo complica la visualización gráfica, sino que también es un problema común en las simulaciones numéricas de sistemas de $N$-cuerpos. El parámetro de suavizado (a menudo denotado como $\epsilon$) actúa como un artificio numérico que mantiene el potencial finito en las cercanías de los cuerpos primarios, simulando de manera aproximada el hecho de que los cuerpos astronómicos tienen dimensiones finitas y no son estrictamente puntuales.

La superficie 3D resultante permite observar visualmente las características topológicas del potencial modificado:
1. **Pozos de potencial profundos:** Alrededor de las posiciones de los cuerpos primarios ($x_1$ y $x_2$), donde la atracción gravitacional domina.
2. **La "cresta" centrífuga:** A medida que nos alejamos del origen, el término centrífugo $-\frac{1}{2}(x^2+y^2)$ domina, haciendo que el potencial disminuya hacia valores negativos más grandes en las regiones exteriores.
3. **Puntos de silla y máximos locales:** En la región entre los cuerpos primarios y en sus alrededores se pueden identificar los puntos de Lagrange colineales ($L_1, L_2, L_3$) como puntos de silla, y los puntos triangulares ($L_4, L_5$) como máximos locales del potencial en el plano $z=0$. Estos puntos críticos son fundamentales para determinar las regiones de exclusión y las superficies de cero velocidad del sistema.

In [92]:
Ng = 100
vmax = 1.5
xs = np.linspace(-vmax, vmax, Ng)
ys = np.linspace(-vmax, vmax, Ng)
Xs, Ys = np.meshgrid(xs, ys)

Zs = 0 * np.ones_like(Xs)
Xs.shape

(100, 100)

In [93]:
Vmods = np.zeros((Ng, Ng))
for iy in range(Ng):
  for ix in range(Ng):
    x = Xs[iy, ix]
    y = Ys[iy, ix]
    z = Zs[iy, ix]

    soft = 0.03
    x1 = -alpha
    x2 = 1 - alpha
    r1 = np.sqrt((x-x1)**2 + y**2 + z**2 + soft)
    r2 = np.sqrt((x-x2)**2 + y**2 + z**2 + soft)
    Vmods[iy, ix] = -(1-alpha)/r1 - alpha/r2 -1/2*(x**2+y**2)

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.plot_surface(Xs, Ys, Vmods, cmap='Spectral')

# Configuración de la vista y etiquetas
ax.view_init(elev=20, azim=330)
ax.set_xlabel('x (unidades canónicas)', fontsize=10)
ax.set_ylabel('y (unidades canónicas)', fontsize=10)
ax.set_zlabel('V_mod', fontsize=10)

# Agregar el título
ax.set_title('Visualización tridimensional del potencial modificado del CRTBP', fontsize=14, pad=20)

plt.tight_layout()
plt.savefig('fig_potencial_modificado.png', dpi=300)
plt.show();

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Surface(z=Vmods, x=Xs, y=Ys)])

fig.update_layout(title='Potencial Modificado en 3D (Plotly)', autosize=False,
                  scene_camera_eye=dict(x=1.86, y=0.18, z=0.61),
                  width=800, height=700,
                  margin=dict(l=65, r=50, b=65, t=90))

fig.show()

### Visualización del Potencial Modificado en el Plano Orbital

El código genera una representación gráfica bidimensional del **potencial modificado** ($V_{mod}$) del Problema Circular Restringido de Tres Cuerpos (CRTBP) evaluado en el plano orbital ($z=0$). Este potencial escalar, definido como la suma del potencial gravitacional newtoniano de los dos cuerpos primarios y el potencial centrífugo asociado a la rotación del marco de referencia, gobierna la aceleración de la partícula de prueba cuando esta se encuentra en reposo en el sistema rotante (Zuluaga, Sección 8.12).

La topografía del potencial revelada por el mapa de colores y las curvas equipotenciales (líneas de contorno) permite identificar analíticamente las características dinámicas fundamentales del sistema:

1. **Pozos de Potencial Gravitacional:** Los valores más negativos (regiones oscuras o frías en el colormap) se localizan en las posiciones de los cuerpos primarios masivos ($m_1$ y $m_2$). Aquí, la interacción gravitacional domina sobre el término centrífugo, creando singularidades teóricas que han sido suavizadas numéricamente en el cálculo mediante el factor `soft` para evitar divergencias infinitas.
2. **Dominio Centrífugo:** A medida que la distancia al baricentro del sistema aumenta, el término del potencial centrífugo ($-\frac{1}{2}(x^2+y^2)$) comienza a dominar, haciendo que el potencial disminuya hacia valores más negativos en las fronteras exteriores del dominio graficado.
3. **Puntos de Equilibrio de Lagrange:** Los puntos críticos de esta superficie (donde el gradiente $\nabla V_{mod} = 0$) corresponden a las posiciones de los cinco puntos de equilibrio de Lagrange. Los puntos colineales ($L_1, L_2, L_3$) aparecen geométricamente como puntos de silla, mientras que los puntos triangulares ($L_4, L_5$) se manifiestan como máximos locales del potencial en el plano $x-y$.

**Observación Física:** Las curvas de nivel negras (equipotenciales) son de vital importancia para la dinámica orbital. Al igualar una curva equipotencial específica a la **Constante de Jacobi** ($C_J$) de una partícula de prueba, se obtienen las **superficies de cero velocidad**. Estas curvas actúan como barreras dinámicas impenetrables que delimitan las regiones de exclusión, restringiendo el movimiento de la partícula de prueba a regiones específicas del espacio permitidas, todo ello sin necesidad de integrar numéricamente las ecuaciones de movimiento (Zuluaga, Sección 8.11).


In [ ]:
plt.figure(figsize=(6,5))

c1 = plt.contourf(Xs, Ys, Vmods, levels=50, cmap='Spectral')
plt.colorbar(c1, label='$V_{mod}$ (unidades canónicas)')
c2 = plt.contour(Xs, Ys, Vmods, levels=5, colors='k', linewidths=1.5)

plt.xlabel('x (unidades canónicas)', fontsize=12)
plt.ylabel('y (unidades canónicas)', fontsize=12)
plt.title('Contornos del Potencial Modificado ($V_{mod}$)', fontsize=14)

plt.axis('equal')
plt.tight_layout()

# Guardar la figura con alta resolución
plt.savefig('fig_potencial_modificado_contornos.png', dpi=300)

plt.show()

### Cálculo de la Constante de Jacobi y Regiones de Exclusión

El código implementa la evaluación de la **Constante de Jacobi** ($C_J$) para la trayectoria de la sonda Lucy en el marco del Problema Circular Restringido de los Tres Cuerpos (CRTBP) y su posterior uso para determinar las **regiones de exclusión** del espacio.

#### 1. Cálculo de la Constante de Jacobi
De acuerdo con la **Sección 8.7** de Zuluaga, la constante de Jacobi es la única cantidad conservada en el CRTBP. El código extrae las posiciones y velocidades de la sonda Lucy a lo largo de su trayectoria numérica y calcula $C_J$ en cada instante utilizando la relación:
$$ C_J = \frac{2(1-\alpha)}{r_1} + \frac{2\alpha}{r_2} + (x^2 + y^2) - v^2 $$
donde $\alpha$ es el parámetro de masa canónico, $r_1$ y $r_2$ son las distancias de la partícula de prueba a los cuerpos primarios (ubicados en $x_1 = -\alpha$ y $x_2 = 1-\alpha$), y $v$ es la rapidez de la partícula en el sistema rotante. Dado que $C_J$ debe ser estrictamente constante, se promedian los valores obtenidos a lo largo de la integración numérica (`CJ = CJs.mean()`) para mitigar posibles errores de truncamiento o redondeo del solucionador numérico.

#### 2. Evaluación de las Regiones de Exclusión
En la **Sección 8.11** de Zuluaga, se establece que la condición física elemental $v^2 \ge 0$ impone una restricción cinemática absoluta sobre el movimiento de la partícula. Despejando $v^2$ de la ecuación de la constante de Jacobi, se obtiene la expresión evaluada en el código para dos puntos de prueba específicos $(x, y, z)$:
$$ v^2 = \frac{2(1-\alpha)}{r_1} + \frac{2\alpha}{r_2} + (x^2 + y^2) - C_J $$

El código evalúa esta expresión en dos coordenadas distintas del plano orbital ($z=0$):
1. **Punto 1:** $(x, y) = (-0.25, -0.6)$
2. **Punto 2:** $(x, y) = (-0.25, -1.6)$

El resultado numérico de $v^2$ dicta la naturaleza de la región en cada punto:
- Si **$v^2 \ge 0$**, el punto pertenece a una **región permitida** y la sonda Lucy, con su $C_J$ actual, tiene la energía cinética suficiente para visitar esa posición.
- Si **$v^2 < 0$**, el punto se encuentra dentro de una **región de exclusión**. Físicamente, esto significa que la sonda nunca podrá alcanzar esa coordenada espacial, ya que requeriría una rapidez imaginaria. Las fronteras donde $v^2 = 0$ definen las **superficies de cero velocidad**, que actúan como barreras dinámicas impenetrables para la partícula de prueba.

In [ ]:
cuerpos_trayectorias = {
    'Hector': (rs_Hector, vs_Hector),
    'Hilda': (rs_Hilda, vs_Hilda),
    'Patroclus': (rs_Patroclus, vs_Patroclus),
    'Lucy': (rs_Lucy, vs_Lucy)
}

resultados_CJs = {}

x1 = -alpha
x2 = 1 - alpha

plt.figure(figsize=(10, 6))

for nombre, (rs, vs) in cuerpos_trayectorias.items():
    xs, ys, zs = rs[:, 0], rs[:, 1], rs[:, 2]
    vmags = np.linalg.norm(vs, axis=1)

    r1 = ((xs - x1)**2 + ys**2 + zs**2)**0.5
    r2 = ((xs - x2)**2 + ys**2 + zs**2)**0.5

    # Definición de la constante de Jacobi
    CJs = 2*(1 - alpha)/r1 + 2*alpha/r2 + (xs**2 + ys**2) - vmags**2
    resultados_CJs[nombre] = CJs

    plt.plot(ts, CJs, label=f'C_J {nombre}')

# Guardamos el promedio de Lucy para celdas posteriores
CJ = resultados_CJs['Lucy'].mean()

plt.title('Evolución temporal de la Constante de Jacobi')
plt.xlabel('Tiempo (unidades canónicas)')
plt.ylabel('C_J')
plt.grid(ls='--')
plt.legend()
plt.tight_layout()
plt.show()

In [98]:
x = -0.25
y = -0.6
z = 0

# Condición de exclusión/aceptación
r1 = ((x-x1)**2 + y**2 + z**2)**0.5
r2 = ((x-x2)**2 + y**2 + z**2)**0.5
v2 = 2*(1-alpha)/r1 + 2*alpha/r2 + x**2 + y**2 - CJ
v2

np.float64(0.8158437268284917)

In [99]:
x = -0.25
y = -1.6
z = 0

# Condición de exclusión/aceptación
r1 = ((x-x1)**2 + y**2 + z**2)**0.5
r2 = ((x-x2)**2 + y**2 + z**2)**0.5
v2 = 2*(1-alpha)/r1 + 2*alpha/r2 + x**2 + y**2 - CJ
v2

np.float64(1.1736346231043084)

### Visualización de las Regiones de Exclusión y Superficies de Cero Velocidad

Para determinar *a priori* las regiones del espacio accesibles para la partícula de prueba (la sonda Lucy) sin necesidad de integrar numéricamente las ecuaciones de movimiento, utilizamos la **Constante de Jacobi** ($C_J$). De acuerdo con la **Sección 8.11** de Zuluaga, la condición física elemental de que el cuadrado de la rapidez sea no negativo ($v^2 \ge 0$) impone una restricción cinemática absoluta sobre el movimiento. Despejando $v^2$ de la definición de $C_J$, obtenemos la desigualdad:

$$ v^2 = \frac{2(1-\alpha)}{r_1} + \frac{2\alpha}{r_2} + x^2 + y^2 - C_J \ge 0 $$

El código evalúa esta expresión en una malla bidimensional del plano orbital ($z=0$), excluyendo mediante una máscara la región inmediatamente adyacente al Sol para evitar las singularidades matemáticas ($r_1 \to 0$) que harían diverger el potencial. Las regiones donde el valor calculado es estrictamente negativo ($v^2 < 0$) se identifican como **regiones de exclusión** (áreas sombreadas en el gráfico), las cuales están prohibidas para la partícula de prueba. El lugar geométrico de los puntos donde la expresión se anula ($v^2 = 0$) define la **superficie de cero velocidad** (línea negra continua).

**Observación Física:** Al superponer la trayectoria numérica de la sonda Lucy sobre este mapa, se verifica que la partícula nunca penetra en las regiones de exclusión. Las superficies de cero velocidad actúan como barreras dinámicas impenetrables o "espejos dinámicos" (Zuluaga, Sección 8.11). Cuando la partícula alcanza esta superficie, su rapidez se anula momentáneamente ($v=0$) antes de ser "reflejada" hacia el interior de la región permitida, confirmando que la dinámica del CRTBP confina estrictamente el movimiento de la partícula de prueba a las zonas de energía permitida.

In [100]:
Ng = 100
vmax = 2.0
xs = np.linspace(-vmax, vmax, Ng)
ys = np.linspace(-vmax, vmax, Ng)
Xs, Ys = np.meshgrid(xs, ys)
Zs = 0 * np.ones_like(Xs)

# Excluye la región muy cercana al Sol
mask = (Xs**2 + Ys**2 > 0.1)  # Evita la singularidad del Sol
Zs_masked = np.ma.masked_where(~mask, Zs)
# Condición de exclusión/aceptación
R1s = ((Xs-x1)**2 + Ys**2 + Zs**2)**0.5
R2s = ((Xs-x2)**2 + Ys**2 + Zs**2)**0.5
V2s = 2*(1-alpha)/R1s + 2*alpha/R2s + Xs**2 + Ys**2 - CJ

In [ ]:
plt.figure(figsize=(6,5))

c1 = plt.contourf(Xs, Ys, V2s,
                  levels=[-100,0], cmap='Spectral')
plt.colorbar(c1)
# Superficie de cero velocidad
c2 = plt.contour(Xs, Ys, V2s,
                 levels=[0], colors='k')

plt.plot(-alpha, 0, 'ro', ms=5)
plt.plot(1-alpha, 0, 'bo', ms=2)
plt.plot(rs_Lucy[:, 0], rs_Lucy[:, 1])

plt.axis('equal')
plt.tight_layout()

## Nota metodológica importante

Todo este bloque de celdas es un **experimento numérico exploratorio**, no un cálculo
riguroso de mecánica celeste. En particular:

- El punto **L1** que usamos aquí **no es un punto de Lagrange calculado exactamente**.
  Se obtiene de la fórmula aproximada de Euler para μ pequeño, válida solo a primer
  orden en μ. No resolvimos la ecuación exacta de equilibrio del CRTBP.
- La "partícula de prueba inspirada en Hilda" **no es Hilda**. Solo tomamos su
  constante de Jacobi real como una referencia de energía para generar una condición
  inicial artificial cerca de L1. Físicamente, Hilda vive en una resonancia 3:2 con
  Júpiter, muy lejos de la región de L1; no hay ninguna dinámica real que la conecte
  con esta trayectoria.
- La búsqueda de parámetros (offset, margen de CJ) es un **barrido tipo curve-fitting**:
  buscamos por fuerza bruta combinaciones que producen "muchas vueltas antes de
  escapar", no derivamos estas condiciones de variedades invariantes estables/inestables
  de una órbita periódica alrededor de L1.
- Por lo tanto, todo lo que sigue debe leerse como **una demostración ilustrativa del
  concepto de "captura balística temporal"**, no como una predicción física validada
  de captura real de un cuerpo tipo Hilda.

Se calcula $x_{L_1}$ con la aproximación de Euler y se evalúa allí $C_J$, comparándolo
con el $C_J$ real de Hilda (obtenido de su vector de estado ya calculado antes en el
notebook). El objetivo es fijar una condición inicial cerca de $L_1$ pero con la
energía de Hilda como restricción: se elige un punto desplazado del lado solar
(offset=0.01) y se despeja la magnitud de velocidad $v$ a partir de
$C_J = 2\Omega(\mathbf{r}) - v^2$, imponiendo que $C_J$ sea el mínimo entre el de
Hilda y un valor ligeramente inferior al de $L_1$ (para forzar un canal angosto si
Hilda tuviera más energía que $L_1$). La dirección de $v$ se elige perpendicular al
eje Sol–Júpiter para inducir un giro alrededor de Júpiter en vez de un cruce
frontal. Se obtiene así una primera trayectoria de prueba, que se integra y grafica
solo para inspección cualitativa.

In [102]:
# Captura temporal cerca de L1, usando la energía (C_J) real de Hilda como referencia

x1, x2 = -alpha, 1 - alpha

def Omega(r):
    x, y, z = r
    r1 = np.sqrt((x-x1)**2 + y**2 + z**2)
    r2 = np.sqrt((x-x2)**2 + y**2 + z**2)
    return (1-alpha)/r1 + alpha/r2 + 0.5*(x**2 + y**2)

def jacobi(r, v):
    return 2*Omega(r) - np.dot(v, v)

# Referencia real: posición y CJ de Hilda
r0_hilda = np.array(resultados_Ys['Ys_153_Hilda'][:3])
v0_hilda = np.array(resultados_Ys['Ys_153_Hilda'][3:])
CJ_hilda_real = jacobi(r0_hilda, v0_hilda)

# Posición de L1 (aprox. de Euler-Lagrange, mu pequeño)
x_L1 = x2 - (alpha/3.0)**(1/3)
r_L1 = np.array([x_L1, 0.0, 0.0])
CJ_L1 = 2*Omega(r_L1)

print(f"CJ real de Hilda:  {CJ_hilda_real:.5f}")
print(f"CJ en L1:           {CJ_L1:.5f}")

# Partícula de prueba "inspirada en Hilda": misma energía (CJ),

offset = 0.01
r0_test = np.array([x_L1 - offset, 0.0, 0.0])

# Para que la energía sea consistente, usamos el CJ real de Hilda (o un poco por debajo
# del de L1 si CJ_hilda_real > CJ_L1, lo cual forzaría el canal cerrado)
CJ_objetivo = min(CJ_hilda_real, CJ_L1 - 0.001)

v_mag2 = 2*Omega(r0_test) - CJ_objetivo
if v_mag2 < 0:
    raise ValueError("CJ objetivo inalcanzable en ese punto: ajusta el offset.")

v_mag = np.sqrt(v_mag2)
# Velocidad casi perpendicular al eje Sol-Júpiter, para inducir el "looping" alrededor
# de Júpiter en vez de un cruce radial directo
v0_test = np.array([0.0, v_mag, 0.0])

print(f"Punto de partida (cerca de L1): {r0_test}")
print(f"Velocidad inicial: {v0_test}")
print(f"CJ resultante: {jacobi(r0_test, v0_test):.5f}")

# Integración
ts_captura = np.linspace(0, 18, 10000)
rs_captura, vs_captura, ris_captura, vis_captura, r1s_captura, r2s_captura = pc.crtbp_solucion(
    alpha, r0_test, v0_test, ts_captura
)

print(f"x mín/máx trayectoria: {rs_captura[:,0].min():.4f} / {rs_captura[:,0].max():.4f}")
print(f"y mín/máx trayectoria: {rs_captura[:,1].min():.4f} / {rs_captura[:,1].max():.4f}")

CJ real de Hilda:  3.02921
CJ en L1:           3.03878
Punto de partida (cerca de L1): [0.92079761 0.         0.        ]
Velocidad inicial: [0.         0.10367823 0.        ]
CJ resultante: 3.02921
x mín/máx trayectoria: -0.8465 / 0.9523
y mín/máx trayectoria: -0.7865 / 0.8732


In [ ]:
# Gráfico estático
plt.figure(figsize=(7, 6))
plt.plot(rs_captura[:,0], rs_captura[:,1], "-c", lw=1.2,
         label="Partícula de prueba (energía ~ Hilda)")
plt.plot(1 - alpha, 0, 'o', color='blue', markersize=10, label='Jupiter')
plt.plot(x_L1, 0, '+', color='black', markersize=12, mew=2, label='L1')
plt.plot(r0_test[0], r0_test[1], '*', color='orange', markersize=12, label='Punto de partida')
plt.xlim(0.8, 1.2)
plt.ylim(-0.2, 0.2)
plt.xlabel('x (unidades canónicas)')
plt.ylabel('y (unidades canónicas)')
plt.title('Captura balística temporal cerca de $L_1$')
plt.grid(ls="--")
plt.axis('equal')
plt.legend()
plt.tight_layout()
plt.show()

Se define un contador heurístico de "vueltas": mientras la partícula está a menos
de 0.15 unidades canónicas de Júpiter, se cuentan los cruces de $y=0$ (cada 2
cruces ≈ 1 vuelta). El objetivo es recorrer una malla de offsets (0.005–0.02) y
márgenes de energía por debajo de $C_J(L_1)$ (0.0001–0.005), integrando cada
combinación en una ventana corta ($t\in[0,10]$), para ubicar qué región de ese
espacio produce más vueltas antes del escape. El resultado es una tabla ordenada
de mayor a menor número de vueltas, que apunta a offset≈0.005 como la zona más
prometedora.

In [104]:
# Buscar parámetros que produzcan más de una vuelta cerca de Júpiter

def cuenta_vueltas_jupiter(rs, x_jup, radio_captura=0.15):
    """Cuenta cruces del eje y=0 mientras la partícula está dentro de
    'radio_captura' de Júpiter (una vuelta completa = ~2 cruces)."""
    dist = np.sqrt((rs[:,0]-x_jup)**2 + rs[:,1]**2)
    cerca = dist < radio_captura
    if not np.any(cerca):
        return 0, 0
    # cruces de y=0 mientras está cerca
    y = rs[:,1]
    signo = np.sign(y)
    cruces = np.where((np.diff(signo) != 0) & cerca[1:])[0]
    return len(cruces) // 2, cerca.sum()  # nº vueltas aprox., nº de puntos "cerca"

resultados_barrido = []

offsets = [0.005, 0.008, 0.01, 0.012, 0.015, 0.02]
margenes_cj = [0.0001, 0.0003, 0.0005, 0.001, 0.002, 0.005]

for off in offsets:
    for marg in margenes_cj:
        r0_test = np.array([x_L1 - off, 0.0, 0.0])
        CJ_objetivo = CJ_L1 - marg   # ahora SIEMPRE ligeramente por debajo de L1 (canal angosto)
        v_mag2 = 2*Omega(r0_test) - CJ_objetivo
        if v_mag2 < 0:
            continue
        v_mag = np.sqrt(v_mag2)
        v0_test = np.array([0.0, v_mag, 0.0])

        ts_prueba = np.linspace(0, 10, 6000)
        rs_p, vs_p, ris_p, vis_p, r1s_p, r2s_p = pc.crtbp_solucion(alpha, r0_test, v0_test, ts_prueba)

        n_vueltas, n_cerca = cuenta_vueltas_jupiter(rs_p, 1-alpha)
        resultados_barrido.append((off, marg, n_vueltas, n_cerca))

# Mostrar los mejores candidatos (más vueltas)
resultados_barrido.sort(key=lambda x: -x[2])
print(f"{'offset':>8} {'margen_CJ':>10} {'vueltas':>8} {'pts_cerca':>10}")
for off, marg, nv, nc in resultados_barrido[:10]:
    print(f"{off:8.4f} {marg:10.4f} {nv:8d} {nc:10d}")

  offset  margen_CJ  vueltas  pts_cerca
  0.0050     0.0050        3       2830
  0.0080     0.0050        1       1365
  0.0050     0.0001        0        910
  0.0050     0.0003        0        952
  0.0050     0.0005        0        993
  0.0050     0.0010        0       1095
  0.0050     0.0020        0       1358
  0.0080     0.0001        0        822
  0.0080     0.0003        0        844
  0.0080     0.0005        0        866


Con offset≈0.005 identificado como punto de partida, se reduce el rango de
búsqueda a offsets muy cercanos (0.004–0.006) y márgenes de energía más grandes
y finos (0.004–0.015), repitiendo el mismo conteo de vueltas. El objetivo es
afinar, dentro de esa vecindad, la combinación exacta (offset, margen) que
maximiza el número de vueltas, resultando en offset=0.005, margen=0.015 como
mejor candidato.

In [105]:
# Explorar alrededor del mejor punto encontrado (offset=0.005)

offsets_finos = [0.004, 0.005, 0.006]
margenes_finos = [0.004, 0.005, 0.006, 0.008, 0.01, 0.015]

resultados_finos = []
for off in offsets_finos:
    for marg in margenes_finos:
        r0_test = np.array([x_L1 - off, 0.0, 0.0])
        CJ_objetivo = CJ_L1 - marg
        v_mag2 = 2*Omega(r0_test) - CJ_objetivo
        if v_mag2 < 0:
            continue
        v_mag = np.sqrt(v_mag2)
        v0_test = np.array([0.0, v_mag, 0.0])

        ts_prueba = np.linspace(0, 10, 6000)
        rs_p, vs_p, ris_p, vis_p, r1s_p, r2s_p = pc.crtbp_solucion(alpha, r0_test, v0_test, ts_prueba)
        n_vueltas, n_cerca = cuenta_vueltas_jupiter(rs_p, 1-alpha)
        resultados_finos.append((off, marg, n_vueltas, n_cerca))

resultados_finos.sort(key=lambda x: -x[2])
print(f"{'offset':>8} {'margen_CJ':>10} {'vueltas':>8} {'pts_cerca':>10}")
for off, marg, nv, nc in resultados_finos[:10]:
    print(f"{off:8.4f} {marg:10.4f} {nv:8d} {nc:10d}")

  offset  margen_CJ  vueltas  pts_cerca
  0.0050     0.0150        6       6000
  0.0040     0.0100        4       3268
  0.0040     0.0150        4       4392
  0.0050     0.0100        4       3346
  0.0060     0.0060        4       5612
  0.0060     0.0080        4       4089
  0.0040     0.0040        3       2682
  0.0040     0.0050        3       2755
  0.0040     0.0060        3       2874
  0.0040     0.0080        3       3186


El barrido anterior solo integró hasta $t=10$, insuficiente para distinguir una
captura temporal real (con escape eventual) de una órbita cuasi-estable que nunca
escapa. Aquí se toma la combinación ganadora y se integra hasta $t=40$,
calculando la distancia a Júpiter en cada paso y buscando el primer instante en
que supera el radio de captura (0.15). El resultado confirma un tiempo de escape
concreto ($t\approx13.02$ según el print posterior), validando que sí se trata de
una captura temporal y no de una órbita atrapada indefinidamente.

In [106]:
# ¿la mejor combinación realmente escapa después de varias vueltas?

off_final = 0.005
marg_final = 0.015

r0_final = np.array([x_L1 - off_final, 0.0, 0.0])
CJ_objetivo_final = CJ_L1 - marg_final
v_mag2_final = 2*Omega(r0_final) - CJ_objetivo_final
v_mag_final = np.sqrt(v_mag2_final)
v0_final = np.array([0.0, v_mag_final, 0.0])

print(f"CJ objetivo: {CJ_objetivo_final:.5f}  (CJ en L1: {CJ_L1:.5f})")
print(f"v0: {v0_final}")

# Integramos MUCHO más tiempo para ver si eventualmente escapa
ts_largo = np.linspace(0, 40, 20000)
rs_largo, vs_largo, ris_largo, vis_largo, r1s_largo, r2s_largo = pc.crtbp_solucion(
    alpha, r0_final, v0_final, ts_largo
)

dist_jup = np.sqrt((rs_largo[:,0]-(1-alpha))**2 + rs_largo[:,1]**2)
print(f"Distancia a Júpiter — mín: {dist_jup.min():.4f}, máx: {dist_jup.max():.4f}")
print(f"x mín/máx: {rs_largo[:,0].min():.4f} / {rs_largo[:,0].max():.4f}")

# ¿En qué momento (si lo hace) supera radio_captura=0.15?
radio_captura = 0.15
fuera = np.where(dist_jup > radio_captura)[0]
if len(fuera) == 0:
    print("=> La partícula NUNCA escapa en t=[0,40]: es una órbita cuasi-estable, no captura temporal.")
else:
    t_escape = ts_largo[fuera[0]]
    print(f"=> Escapa por primera vez en t = {t_escape:.3f} (unidades canónicas)")

CJ objetivo: 3.02378  (CJ en L1: 3.03878)
v0: [0.         0.12401223 0.        ]
Distancia a Júpiter — mín: 0.0007, máx: 2.6466
x mín/máx: -1.6398 / 1.6963
=> Escapa por primera vez en t = 13.023 (unidades canónicas)


Se reconstruye la condición inicial ya verificada (offset=0.005, margen=0.015) y
se integra hasta $t=18$ (un poco más allá del escape confirmado) para capturar
tanto la fase de captura como el inicio del alejamiento. Se generan dos figuras:
una vista general de toda la trayectoria y un zoom de la fase previa a $t=13.5$
centrado en Júpiter, junto con el conteo final de vueltas y el rango de $x$
alcanzado.

In [107]:
off_final = 0.005
marg_final = 0.015

x_L1 = (1-alpha) - (alpha/3.0)**(1/3)
r0_test = np.array([x_L1 - off_final, 0.0, 0.0])

CJ_objetivo = CJ_L1 - marg_final
v_mag = np.sqrt(2*Omega(r0_test) - CJ_objetivo)
v0_test = np.array([0.0, v_mag, 0.0])

print(f"CJ objetivo: {CJ_objetivo:.5f}  (CJ en L1: {CJ_L1:.5f})")
print(f"Punto de partida: {r0_test}")
print(f"Velocidad inicial: {v0_test}")

# Integramos un poco más allá del escape confirmado (t=13.02) para ver
# cómo se aleja después de las vueltas
ts_captura = np.linspace(0, 18, 12000)
rs_captura, vs_captura, ris_captura, vis_captura, r1s_captura, r2s_captura = pc.crtbp_solucion(
    alpha, r0_test, v0_test, ts_captura
)

dist_jup = np.sqrt((rs_captura[:,0]-(1-alpha))**2 + rs_captura[:,1]**2)
n_vueltas, n_cerca = cuenta_vueltas_jupiter(rs_captura, 1-alpha)
print(f"Vueltas cerca de Júpiter: {n_vueltas}")
print(f"x mín/máx: {rs_captura[:,0].min():.4f} / {rs_captura[:,0].max():.4f}")

CJ objetivo: 3.02378  (CJ en L1: 3.03878)
Punto de partida: [0.92579761 0.         0.        ]
Velocidad inicial: [0.         0.12401223 0.        ]
Vueltas cerca de Júpiter: 7
x mín/máx: -1.2161 / 1.2631


In [ ]:
# Gráfico 1: vista general (toda la trayectoria, incluyendo el escape)
plt.figure(figsize=(8, 7))
plt.plot(rs_captura[:,0], rs_captura[:,1], "-c", lw=1.0, label="Partícula de prueba")
plt.plot(1-alpha, 0, 'o', color='blue', markersize=10, label='Jupiter')
plt.plot(x_L1, 0, '+', color='black', markersize=12, mew=2, label='L1')
plt.plot(r0_test[0], r0_test[1], '*', color='orange', markersize=12, label='Punto de partida')
plt.xlabel('x (unidades canónicas)')
plt.ylabel('y (unidades canónicas)')
plt.title(f'Captura temporal: {n_vueltas} vueltas cerca de Júpiter, luego escape')
plt.grid(ls="--")
plt.axis('equal')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 2: zoom en la fase de captura (antes del escape)
mask_captura = ts_captura < 13.5
plt.figure(figsize=(7, 6))
plt.plot(rs_captura[mask_captura,0], rs_captura[mask_captura,1], "-c", lw=1.2,
         label=f"Fase de captura ({n_vueltas} vueltas)")
plt.plot(1-alpha, 0, 'o', color='blue', markersize=10, label='Jupiter')
plt.plot(x_L1, 0, '+', color='black', markersize=12, mew=2, label='L1')
plt.xlim(1-alpha-0.2, 1-alpha+0.2)
plt.ylim(-0.2, 0.2)
plt.xlabel('x (unidades canónicas)')
plt.ylabel('y (unidades canónicas)')
plt.title('Zoom: fase de captura antes del escape')
plt.grid(ls="--")
plt.axis('equal')
plt.legend()
plt.tight_layout()
plt.savefig('fig_ZoomFaseDeCaptura.png', dpi=300)
plt.show()

Se anima la misma trayectoria  en dos segmentos con distinta escala:
ejes fijos cerca de Júpiter durante la fase de captura ($t<13.5$) y ejes que se
ajustan al rango completo durante la fase de escape. Es puramente una
herramienta de visualización del resultado ya obtenido, sin cálculo adicional.

In [ ]:
from matplotlib.animation import FuncAnimation
plt.rcParams['animation.html'] = 'jshtml'
plt.rcParams['animation.embed_limit'] = 150

# Fase de captura: t en [0, 13.5], con zoom cerca de Júpiter
idx_captura = np.where(ts_captura < 13.5)[0]
idx_captura = idx_captura[::max(1, len(idx_captura)//250)]

# Fase de escape: t en [13.5, 18], vista general
idx_escape = np.where(ts_captura >= 13.5)[0]
idx_escape = idx_escape[::max(1, len(idx_escape)//150)]

fig1, ax1 = plt.subplots(figsize=(8, 7))
ax1.plot(1-alpha, 0, 'o', color='blue', markersize=10, label='Jupiter')
ax1.plot(x_L1, 0, '+', color='black', markersize=12, mew=2, label='L1')
ax1.grid(ls="--")
ax1.set_xlabel('x (unidades canónicas)')
ax1.set_ylabel('y (unidades canónicas)')
ax1.legend(loc='upper right')

linea1, = ax1.plot([], [], "-c", lw=1.0)
punto1, = ax1.plot([], [], "o", color="darkcyan", markersize=6)

n1 = len(idx_captura)
n2 = len(idx_escape)

def init1():
    ax1.set_xlim(1-alpha-0.2, 1-alpha+0.2)
    ax1.set_ylim(-0.2, 0.2)
    ax1.set_title('Fase de captura: vueltas cerca de Júpiter')
    return linea1, punto1

def update1(frame):
    if frame < n1:
        i = idx_captura[frame]
        linea1.set_data(rs_captura[:i+1, 0], rs_captura[:i+1, 1])
        punto1.set_data([rs_captura[i, 0]], [rs_captura[i, 1]])
        ax1.set_xlim(1-alpha-0.2, 1-alpha+0.2)
        ax1.set_ylim(-0.2, 0.2)
        ax1.set_title('Fase de captura: vueltas cerca de Júpiter')
    else:
        j = idx_escape[frame - n1]
        linea1.set_data(rs_captura[:j+1, 0], rs_captura[:j+1, 1])
        punto1.set_data([rs_captura[j, 0]], [rs_captura[j, 1]])
        ax1.set_xlim(rs_captura[:,0].min()-0.1, rs_captura[:,0].max()+0.1)
        ax1.set_ylim(rs_captura[:,1].min()-0.1, rs_captura[:,1].max()+0.1)
        ax1.set_title('Fase de escape: la partícula abandona la vecindad de Júpiter')
    return linea1, punto1

anim1 = FuncAnimation(fig1, update1, init_func=init1, frames=n1+n2,
                       interval=50, blit=False)
plt.close(fig1)
anim1

Como control de calidad numérico, se evalúa $C_J$ en cada punto de la trayectoria
final y se compara su valor inicial, final y la deriva máxima absoluta a lo
largo de la integración. Una deriva pequeña respalda que el número de vueltas y
el tiempo de escape reportados no son artefactos de error numérico acumulado por
el integrador.

In [111]:
# Verificación de conservación de CJ
CJ_serie = jacobi(rs_captura.T, vs_captura.T) if False else None
CJ_a_lo_largo = 2*np.array([Omega(r) for r in rs_captura]) - np.linalg.norm(vs_captura, axis=1)**2
print(f"CJ inicial: {CJ_a_lo_largo[0]:.6f}, CJ final: {CJ_a_lo_largo[-1]:.6f}, deriva máx: {np.abs(CJ_a_lo_largo - CJ_a_lo_largo[0]).max():.2e}")

CJ inicial: 3.023780, CJ final: 3.023760, deriva máx: 2.04e-05


## Conclusión general del experimento

El experimento parte de la energía real de Hilda como referencia y, mediante un
barrido numérico de offset y margen respecto a $C_J(L_1)$ (aproximado), encuentra
una condición inicial que produce varias vueltas alrededor de Júpiter antes de
escapar en $t\approx13$, comportamiento confirmado a tiempo largo y respaldado
por una deriva mínima de $C_J$. Aun así, el resultado es una **ilustración
cualitativa** de captura balística temporal, no una predicción física sobre
Hilda: no se usaron variedades invariantes ni un $L_1$ exacto, y la condición
inicial se ajustó por búsqueda directa de una métrica geométrica, no por
derivación dinámica rigurosa.